In [18]:
from ultralytics import YOLO
model = YOLO('data/model/classify_yolo_nano_v2.pt')

# 2. Get the class names dictionary
class_dict = model.names
print(f"Detected Classes: {class_dict}")

Detected Classes: {0: 'Blue_Yellow', 1: 'Bran', 2: 'Brown_Orange_Overlay', 3: 'Brown_Orange_Small', 4: 'Green_Yellow', 5: 'Red_Yellow', 6: 'Wheatberry'}


In [34]:
import cv2
import imagehash
from PIL import Image

def compute_phash(image):
    # Convert BGR (OpenCV) to RGB, then to PIL Image
    pil_img = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    return imagehash.phash(pil_img)


def is_good_roi(
    roi,
    min_size=60,
    max_aspect_ratio=1.5,
    min_sharpness=100,
    min_mean_brightness=40,
    max_mean_brightness=220,
):
    h, w = roi.shape[:2]
    aspect_ratio = max(h / w, w / h)
    if h < min_size or w < min_size:
        return False
    if aspect_ratio > max_aspect_ratio:
        return False

    # Sharpness
    sharpness = cv2.Laplacian(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    if sharpness < min_sharpness:
        return False

    # Brightness
    mean_brightness = roi.mean()
    if not (min_mean_brightness <= mean_brightness <= max_mean_brightness):
        return False

    return True